In [0]:
CATALOG = "ml_projects"
SCHEMA = "fraud_project"
NAMESPACE = f"{CATALOG}.{SCHEMA}"

spark.sql(f"""
CREATE CATALOG IF NOT EXISTS {CATALOG}
COMMENT 'Data and models for machine learning projects'
""")

spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {NAMESPACE}
COMMENT 'Government application fraud detection project'
""")

print(f"Project schema is ready: {NAMESPACE}")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {NAMESPACE}.applications (
    application_id STRING,
    application_timestamp TIMESTAMP,
    requested_amount DOUBLE,
    verified_monthly_income DOUBLE,
    income_discrepancy_ratio DOUBLE,
    applications_last_90_days DOUBLE,
    document_inconsistency_count DOUBLE,
    shared_bank_account_count DOUBLE
)
USING DELTA
COMMENT 'Application features available when a prediction is requested'
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {NAMESPACE}.labels (
    application_id STRING,
    fraud_label INT,
    label_available_at TIMESTAMP
)
USING DELTA
COMMENT 'Investigation outcomes and when they became available'
""")

print("Applications and labels tables are ready.")

In [0]:
display(
    spark.sql(f"SHOW TABLES IN {NAMESPACE}")
)

display(
    spark.sql(f"""
        SELECT 'applications' AS table_name, COUNT(*) AS row_count
        FROM {NAMESPACE}.applications

        UNION ALL

        SELECT 'labels' AS table_name, COUNT(*) AS row_count
        FROM {NAMESPACE}.labels
    """)
)

In [0]:
FEATURES = [
    "requested_amount",
    "verified_monthly_income",
    "income_discrepancy_ratio",
    "applications_last_90_days",
    "document_inconsistency_count",
    "shared_bank_account_count",
]

spark.conf.set("spark.sql.session.timeZone", "UTC")

In [0]:
import numpy as np
import pandas as pd


def generate_applications(
    batch_id,
    start_date,
    end_date,
    n_rows=2000,
    seed=42,
    amount_multiplier=1.0,
    repeat_rate=0.4,
    missing_income_rate=0.02,
    shared_account_weight=0.7,
):
    rng = np.random.default_rng(seed)

    income = np.round(
        rng.lognormal(np.log(1200), 0.55, n_rows),
        2,
    )

    discrepancy = np.where(
        rng.random(n_rows) < 0.12,
        rng.uniform(0.5, 1.5, n_rows),
        rng.beta(1, 10, n_rows),
    )

    repeats = rng.poisson(repeat_rate, n_rows)
    document_issues = rng.poisson(0.3, n_rows)
    shared_accounts = rng.poisson(0.15, n_rows)

    requested_amount = np.round(
        rng.lognormal(np.log(1400), 0.5, n_rows)
        * amount_multiplier,
        2,
    )

    # Fictional mechanism used only to generate labels.
    hidden_risk = (
        -4.0
        + 2.5 * discrepancy
        + 0.6 * document_issues
        + 0.4 * repeats
        + shared_account_weight * shared_accounts
    )

    hidden_probability = 1 / (1 + np.exp(-hidden_risk))
    fraud_labels = rng.binomial(1, hidden_probability)

    income[rng.random(n_rows) < missing_income_rate] = np.nan

    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    duration_seconds = int((end - start).total_seconds())

    if duration_seconds <= 0:
        raise ValueError("end_date must be after start_date.")

    timestamps = start + pd.to_timedelta(
        rng.integers(0, duration_seconds, n_rows),
        unit="s",
    )

    ids = [
        f"{batch_id}-{i:06d}" for i in range(n_rows)
    ]

    applications = pd.DataFrame({
        "application_id": ids,
        "application_timestamp": timestamps,
        "requested_amount": requested_amount,
        "verified_monthly_income": income,
        "income_discrepancy_ratio": discrepancy,
        "applications_last_90_days": repeats.astype(float),
        "document_inconsistency_count": document_issues.astype(float),
        "shared_bank_account_count": shared_accounts.astype(float),
    })

    labels = pd.DataFrame({
        "application_id": ids,
        "fraud_label": fraud_labels.astype("int32"),
        "label_available_at": timestamps + pd.Timedelta(days=7),
    })

    return applications, labels

In [0]:
baseline_applications, baseline_labels = generate_applications(
    batch_id="BASE",
    start_date="2024-10-01",
    end_date="2024-12-25",
    n_rows=5000,
    seed=42,
)

display(baseline_applications.head(10))

display(pd.DataFrame({
    "measurement": [
        "Application count",
        "Fraud proportion",
        "Missing income proportion",
    ],
    "value": [
        len(baseline_applications),
        baseline_labels["fraud_label"].mean(),
        baseline_applications[
            "verified_monthly_income"
        ].isna().mean(),
    ],
}))

In [0]:
def insert_new_records(pdf, table_name, temp_view):
    target = f"{NAMESPACE}.{table_name}"
    target_schema = spark.table(target).schema

    incoming = spark.createDataFrame(
        pdf,
        schema=target_schema,
    )

    if table_name == "applications":
        incoming = incoming.replace(
            float("nan"),
            None,
            subset=["verified_monthly_income"],
        )

    incoming.createOrReplaceTempView(temp_view)

    spark.sql(f"""
        MERGE INTO {target} AS target
        USING {temp_view} AS source
        ON target.application_id = source.application_id
        WHEN NOT MATCHED THEN INSERT *
    """)


insert_new_records(
    baseline_applications,
    "applications",
    "incoming_applications",
)

insert_new_records(
    baseline_labels,
    "labels",
    "incoming_labels",
)

display(spark.sql(f"""
    SELECT 'applications' AS table_name, COUNT(*) AS rows
    FROM {NAMESPACE}.applications
    WHERE application_id LIKE 'BASE-%'

    UNION ALL

    SELECT 'labels' AS table_name, COUNT(*) AS rows
    FROM {NAMESPACE}.labels
    WHERE application_id LIKE 'BASE-%'
"""))

In [0]:
import numpy as np
import pandas as pd
import mlflow
from mlflow.deployments import get_deploy_client
from pyspark.sql import functions as F

CATALOG = "ml_projects"
SCHEMA = "fraud_project"
NAMESPACE = f"{CATALOG}.{SCHEMA}"

APPLICATIONS_TABLE = f"{NAMESPACE}.applications"
LABELS_TABLE = f"{NAMESPACE}.labels"
PREDICTIONS_TABLE = f"{NAMESPACE}.predictions"
ENDPOINT_NAME = "government-fraud-api"
MODEL_VERSION = "3"

FEATURES = [
    "requested_amount",
    "verified_monthly_income",
    "income_discrepancy_ratio",
    "applications_last_90_days",
    "document_inconsistency_count",
    "shared_bank_account_count",
]

def merge_pdf(pdf, table_name, view_name):
    """Добавляет только новые application_id, не создавая дубликаты."""
    target = spark.table(table_name)
    incoming = spark.createDataFrame(pdf, schema=target.schema)
    incoming.createOrReplaceTempView(view_name)

    spark.sql(f"""
        MERGE INTO {table_name} AS target
        USING {view_name} AS source
        ON target.application_id = source.application_id
        WHEN NOT MATCHED THEN INSERT *
    """)

def make_month(batch_id, start_date, end_date, n_rows, seed, amount_multiplier):
    rng = np.random.default_rng(seed)

    income = np.round(
        rng.lognormal(mean=np.log(1200), sigma=0.55, size=n_rows), 2
    )
    discrepancy = np.where(
        rng.random(n_rows) < 0.12,
        rng.uniform(0.5, 1.5, n_rows),
        rng.beta(1, 10, n_rows),
    )
    repeats = rng.poisson(0.4, n_rows)
    document_issues = rng.poisson(0.3, n_rows)
    shared_accounts = rng.poisson(0.15, n_rows)
    amounts = np.round(
        rng.lognormal(mean=np.log(1400), sigma=0.5, size=n_rows)
        * amount_multiplier,
        2,
    )

    hidden_risk = (
        -4
        + 2.5 * discrepancy
        + 0.6 * document_issues
        + 0.4 * repeats
        + 0.7 * shared_accounts
    )
    fraud_probability = 1 / (1 + np.exp(-hidden_risk))
    fraud_label = rng.binomial(1, fraud_probability).astype("int32")

    income[rng.random(n_rows) < 0.02] = np.nan

    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    seconds = int((end - start).total_seconds())

    timestamps = start + pd.to_timedelta(
        rng.integers(0, seconds, n_rows), unit="s"
    )
    ids = [f"{batch_id}-{i:05d}" for i in range(n_rows)]

    apps = pd.DataFrame({
        "application_id": ids,
        "application_timestamp": timestamps,
        "requested_amount": amounts,
        "verified_monthly_income": income,
        "income_discrepancy_ratio": discrepancy,
        "applications_last_90_days": repeats.astype(float),
        "document_inconsistency_count": document_issues.astype(float),
        "shared_bank_account_count": shared_accounts.astype(float),
    })

    labels = pd.DataFrame({
        "application_id": ids,
        "fraud_label": fraud_label,
        "label_available_at": timestamps + pd.Timedelta(days=7),
    })

    return apps, labels

serving_client = get_deploy_client("databricks")

for month_number in range(1, 13):
    start = pd.Timestamp(2025, month_number, 1)
    end = start + pd.offsets.MonthBegin(1)

    # После июня запросы становятся заметно крупнее, чтобы показать drift.
    amount_multiplier = 1.0 if month_number <= 6 else 1.8

    batch_id = f"SIM2025{month_number:02d}"

    apps_pdf, labels_pdf = make_month(
        batch_id=batch_id,
        start_date=start,
        end_date=end,
        n_rows=250,
        seed=100 + month_number,
        amount_multiplier=amount_multiplier,
    )

    # Spark-таблицы уже существуют; добавляем только новые application_id.
    apps_pdf = apps_pdf.astype(object).where(pd.notna(apps_pdf), None)
    merge_pdf(apps_pdf, APPLICATIONS_TABLE, f"apps_{batch_id}")
    merge_pdf(labels_pdf, LABELS_TABLE, f"labels_{batch_id}")

    request_pdf = apps_pdf[["application_id"] + FEATURES].copy()
    request_pdf = request_pdf.astype(object).where(pd.notna(request_pdf), None)

    payload = {
        "dataframe_split": {
            "index": list(range(len(request_pdf))),
            "columns": request_pdf.columns.tolist(),
            "data": request_pdf.values.tolist(),
        }
    }

    response = serving_client.predict(
        endpoint=ENDPOINT_NAME,
        inputs=payload,
    )

    predictions_pdf = pd.DataFrame(response["predictions"])
    predictions_sdf = (
        spark.createDataFrame(predictions_pdf)
        .withColumn(
            "prediction_timestamp",
            F.lit(end.to_pydatetime()).cast("timestamp")
        )
        .withColumn("model_version", F.lit(MODEL_VERSION))
        .select(
            "application_id",
            "prediction_timestamp",
            "fraud_probability",
            "review_required",
            "model_build_id",
            "model_version",
        )
    )

    predictions_sdf.write.mode("append").saveAsTable(PREDICTIONS_TABLE)

    print(
        f"{batch_id}: {len(apps_pdf)} applications scored "
        f"by endpoint {ENDPOINT_NAME}"
    )

In [0]:
import pandas as pd

monthly_cutoffs = pd.date_range(
    start="2025-02-01",
    periods=12,
    freq="MS",
)

for cutoff in monthly_cutoffs:
    as_of = cutoff.strftime("%Y-%m-%dT%H:%M:%S")

    model_version = dbutils.notebook.run(
        "./04_retrain",
        timeout_seconds=0,
        arguments={"as_of": as_of},
    )

    print(f"{cutoff:%Y-%m}: trained and registered model version {model_version}")